<a href="https://colab.research.google.com/github/akshaypan/Banking/blob/main/banking_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install Flask pytest

In [4]:

import sqlite3
import hashlib
import secrets
from decimal import Decimal, InvalidOperation

from flask import Flask, request, jsonify
from werkzeug.security import generate_password_hash, check_password_hash

app = Flask(__name__)

# In-memory database: suitable for a Colab demonstration.
# Data resets when the runtime restarts.
DATABASE = ":memory:"
db = sqlite3.connect(DATABASE, check_same_thread=False)
db.row_factory = sqlite3.Row
db.execute("PRAGMA foreign_keys = ON")

db.executescript("""
CREATE TABLE users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT UNIQUE COLLATE NOCASE NOT NULL,
    password_hash TEXT NOT NULL
);

CREATE TABLE accounts (
    user_id INTEGER PRIMARY KEY REFERENCES users(id),
    balance_paise INTEGER NOT NULL DEFAULT 0 CHECK(balance_paise >= 0)
);

CREATE TABLE transactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sender_id INTEGER NOT NULL REFERENCES users(id),
    receiver_id INTEGER NOT NULL REFERENCES users(id),
    amount_paise INTEGER NOT NULL CHECK(amount_paise > 0),
    created_at TEXT DEFAULT CURRENT_TIMESTAMP,
    CHECK(sender_id != receiver_id)
);

CREATE TABLE api_tokens (
    token_hash TEXT PRIMARY KEY,
    user_id INTEGER NOT NULL REFERENCES users(id)
);

CREATE INDEX idx_sender ON transactions(sender_id, id DESC);
CREATE INDEX idx_receiver ON transactions(receiver_id, id DESC);
""")
db.commit()


def money(paise):
    return f"{paise // 100}.{paise % 100:02d}"


def parse_amount(value):
    if isinstance(value, bool) or not isinstance(value, (str, int, float)):
        raise ValueError("Enter a valid amount.")

    try:
        amount = Decimal(str(value))
    except (InvalidOperation, ValueError):
        raise ValueError("Enter a valid amount.")

    if not amount.is_finite() or amount <= 0:
        raise ValueError("Amount must be positive.")

    if amount.as_tuple().exponent < -2:
        raise ValueError("Maximum two decimal places allowed.")

    paise = amount * 100
    if paise != paise.to_integral_value():
        raise ValueError("Invalid amount.")

    if paise > 9_000_000_000_000_000:
        raise ValueError("Amount too large.")

    return int(paise)


def issue_token(user_id):
    token = secrets.token_urlsafe(32)
    token_hash = hashlib.sha256(token.encode()).hexdigest()
    db.execute(
        "INSERT INTO api_tokens VALUES (?, ?)",
        (token_hash, user_id)
    )
    return token


def get_user():
    parts = request.headers.get("Authorization", "").split()
    if len(parts) != 2 or parts[0].lower() != "bearer":
        return None

    token_hash = hashlib.sha256(parts[1].encode()).hexdigest()
    return db.execute("""
        SELECT u.id, u.username
        FROM users u
        JOIN api_tokens t ON t.user_id = u.id
        WHERE t.token_hash = ?
    """, (token_hash,)).fetchone()


def json_data():
    data = request.get_json(silent=True)
    return data if isinstance(data, dict) else None


@app.post("/register")
def register():
    data = json_data()
    if data is None:
        return jsonify(error="Send valid JSON."), 400

    username = data.get("username")
    password = data.get("password")

    if not isinstance(username, str):
        return jsonify(error="Invalid username."), 400

    username = username.strip()
    if not 3 <= len(username) <= 30:
        return jsonify(error="Username must be 3-30 characters."), 400

    if not username.replace("_", "").isalnum():
        return jsonify(error="Use letters, numbers, underscores only."), 400

    if not isinstance(password, str) or not 12 <= len(password) <= 1024:
        return jsonify(error="Password must contain 12-1024 characters."), 400

    try:
        db.execute("BEGIN IMMEDIATE")
        cursor = db.execute(
            "INSERT INTO users(username,password_hash) VALUES (?,?)",
            (username, generate_password_hash(password))
        )
        uid = cursor.lastrowid
        db.execute("INSERT INTO accounts(user_id) VALUES (?)", (uid,))
        token = issue_token(uid)
        db.commit()
        return jsonify(
            message="Registration successful",
            username=username,
            access_token=token
        ), 201

    except sqlite3.IntegrityError:
        db.rollback()
        return jsonify(error="Username already exists."), 409
    except Exception:
        db.rollback()
        app.logger.exception("Registration failed")
        return jsonify(error="Registration failed."), 500


@app.post("/login")
def login():
    data = json_data()
    if data is None:
        return jsonify(error="Send valid JSON."), 400

    username = data.get("username")
    password = data.get("password")

    if not isinstance(username, str) or not isinstance(password, str):
        return jsonify(error="Username and password are required."), 400

    user = db.execute(
        "SELECT * FROM users WHERE username=? COLLATE NOCASE",
        (username.strip(),)
    ).fetchone()

    if not user or not check_password_hash(user["password_hash"], password):
        return jsonify(error="Invalid username or password."), 401

    try:
        token = issue_token(user["id"])
        db.commit()
    except sqlite3.Error:
        db.rollback()
        return jsonify(error="Login failed."), 500

    return jsonify(message="Login successful", access_token=token)


@app.post("/logout")
def logout():
    parts = request.headers.get("Authorization", "").split()
    if len(parts) != 2 or parts[0].lower() != "bearer":
        return jsonify(error="Authentication required."), 401

    token_hash = hashlib.sha256(parts[1].encode()).hexdigest()
    cursor = db.execute(
        "DELETE FROM api_tokens WHERE token_hash=?", (token_hash,)
    )
    db.commit()

    if cursor.rowcount == 0:
        return jsonify(error="Invalid token."), 401

    return jsonify(message="Logged out")


@app.get("/balance")
def balance():
    user = get_user()
    if user is None:
        return jsonify(error="Authentication required."), 401

    row = db.execute(
        "SELECT balance_paise FROM accounts WHERE user_id=?",
        (user["id"],)
    ).fetchone()

    return jsonify(
        username=user["username"],
        balance=money(row["balance_paise"]),
        currency="INR"
    )


@app.post("/transfer")
def transfer():
    user = get_user()
    if user is None:
        return jsonify(error="Authentication required."), 401

    data = json_data()
    if data is None:
        return jsonify(error="Send valid JSON."), 400

    receiver_name = data.get("receiver")
    if not isinstance(receiver_name, str):
        return jsonify(error="Receiver is required."), 400

    try:
        amount = parse_amount(data.get("amount"))
    except ValueError as exc:
        return jsonify(error=str(exc)), 400

    try:
        db.execute("BEGIN IMMEDIATE")

        receiver = db.execute(
            "SELECT id, username FROM users WHERE username=? COLLATE NOCASE",
            (receiver_name.strip(),)
        ).fetchone()

        if receiver is None:
            db.rollback()
            return jsonify(error="Receiver not found."), 404

        if receiver["id"] == user["id"]:
            db.rollback()
            return jsonify(error="Cannot transfer to yourself."), 400

        debit = db.execute("""
            UPDATE accounts
            SET balance_paise = balance_paise - ?
            WHERE user_id = ? AND balance_paise >= ?
        """, (amount, user["id"], amount))

        if debit.rowcount != 1:
            db.rollback()
            return jsonify(error="Insufficient balance."), 400

        credit = db.execute("""
            UPDATE accounts
            SET balance_paise = balance_paise + ?
            WHERE user_id = ?
        """, (amount, receiver["id"]))

        if credit.rowcount != 1:
            db.rollback()
            return jsonify(error="Receiver account not found."), 404

        cursor = db.execute("""
            INSERT INTO transactions(sender_id,receiver_id,amount_paise)
            VALUES (?,?,?)
        """, (user["id"], receiver["id"], amount))

        tx_id = cursor.lastrowid
        db.commit()

        return jsonify(
            message="Transfer successful",
            transaction_id=tx_id,
            sender=user["username"],
            receiver=receiver["username"],
            amount=money(amount),
            currency="INR"
        ), 201

    except sqlite3.Error:
        if db.in_transaction:
            db.rollback()
        app.logger.exception("Transfer failed")
        return jsonify(error="Transfer failed."), 500


@app.get("/transactions")
def transactions():
    user = get_user()
    if user is None:
        return jsonify(error="Authentication required."), 401

    try:
        limit = int(request.args.get("limit", "20"))
        offset = int(request.args.get("offset", "0"))
    except ValueError:
        return jsonify(error="Invalid pagination."), 400

    if not 1 <= limit <= 100 or offset < 0:
        return jsonify(error="Limit must be 1-100; offset must be nonnegative."), 400

    rows = db.execute("""
        SELECT t.id, s.username AS sender, r.username AS receiver,
               t.amount_paise, t.created_at,
               CASE WHEN t.sender_id=? THEN 'sent' ELSE 'received' END AS direction
        FROM transactions t
        JOIN users s ON s.id=t.sender_id
        JOIN users r ON r.id=t.receiver_id
        WHERE t.sender_id=? OR t.receiver_id=?
        ORDER BY t.id DESC LIMIT ? OFFSET ?
    """, (user["id"], user["id"], user["id"], limit, offset)).fetchall()

    return jsonify(transactions=[{
        "transaction_id": r["id"],
        "sender": r["sender"],
        "receiver": r["receiver"],
        "amount": money(r["amount_paise"]),
        "currency": "INR",
        "direction": r["direction"],
        "created_at": r["created_at"]
    } for r in rows])


@app.get("/health")
def health():
    return jsonify(status="ok")


# Test helper: fund a demo account directly.
# This is not a public API endpoint.
def demo_fund(username, amount_rupees):
    amount = parse_amount(amount_rupees)
    db.execute("BEGIN IMMEDIATE")
    try:
        cursor = db.execute("""
            UPDATE accounts SET balance_paise=balance_paise+?
            WHERE user_id=(SELECT id FROM users WHERE username=? COLLATE NOCASE)
        """, (amount, username))
        if cursor.rowcount != 1:
            raise ValueError("User not found.")
        db.commit()
    except Exception:
        db.rollback()
        raise


print("Application created successfully.")
print("Database tables:", [
    row["name"] for row in
    db.execute("SELECT name FROM sqlite_master WHERE type='table'")
])


Application created successfully.
Database tables: ['users', 'sqlite_sequence', 'accounts', 'transactions', 'api_tokens']


In [8]:

import gradio as gr
import pandas as pd
import sqlite3

# Uses the existing SQLite connection named db.
# Do NOT recreate your Flask app in this cell.

def get_table_data(table_name):
    """Read dashboard data without exposing password hashes or API tokens."""

    allowed_tables = {"users", "accounts", "transactions"}

    if table_name not in allowed_tables:
        return pd.DataFrame({"Error": ["Invalid table"]})

    try:
        columns = [
            row[1]
            for row in db.execute(
                f"PRAGMA table_info({table_name})"
            ).fetchall()
        ]

        if not columns:
            return pd.DataFrame({
                "Error": [f"Table '{table_name}' not found"]
            })

        # Never display password hashes or token-related fields.
        safe_columns = [
            c for c in columns
            if not any(
                word in c.lower()
                for word in ["password", "token", "secret"]
            )
        ]

        if not safe_columns:
            return pd.DataFrame({"Info": ["No safe columns available"]})

        query = (
            f"SELECT {', '.join(safe_columns)} "
            f"FROM {table_name} LIMIT 500"
        )

        rows = db.execute(query).fetchall()
        return pd.DataFrame(rows, columns=safe_columns)

    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})


def dashboard_summary():
    try:
        user_count = db.execute(
            "SELECT COUNT(*) FROM users"
        ).fetchone()[0]

        account_count = db.execute(
            "SELECT COUNT(*) FROM accounts"
        ).fetchone()[0]

        transaction_count = db.execute(
            "SELECT COUNT(*) FROM transactions"
        ).fetchone()[0]

        return (
            f"### Wallet Administration\n"
            f"- Registered users: **{user_count}**\n"
            f"- Accounts: **{account_count}**\n"
            f"- Transactions: **{transaction_count}**"
        )

    except Exception as e:
        return f"Unable to load dashboard summary: {e}"


def refresh_users():
    return dashboard_summary(), get_table_data("users")


def refresh_accounts():
    return dashboard_summary(), get_table_data("accounts")


def refresh_transactions():
    return dashboard_summary(), get_table_data("transactions")


with gr.Blocks(title="Wallet Admin Dashboard") as admin_dashboard:
    gr.Markdown("# Wallet Admin Dashboard")
    gr.Markdown(
        "Administrative overview of users, accounts and transaction records."
    )

    summary = gr.Markdown(dashboard_summary())

    with gr.Tab("Users"):
        users_table = gr.Dataframe(
            value=get_table_data("users"),
            label="Registered Users",
            interactive=False
        )
        gr.Button("Refresh Users").click(
            refresh_users,
            outputs=[summary, users_table]
        )

    with gr.Tab("Accounts"):
        accounts_table = gr.Dataframe(
            value=get_table_data("accounts"),
            label="Wallet Accounts",
            interactive=False
        )
        gr.Button("Refresh Accounts").click(
            refresh_accounts,
            outputs=[summary, accounts_table]
        )

    with gr.Tab("Transactions"):
        transactions_table = gr.Dataframe(
            value=get_table_data("transactions"),
            label="Transaction Records",
            interactive=False
        )
        gr.Button("Refresh Transactions").click(
            refresh_transactions,
            outputs=[summary, transactions_table]
        )

admin_dashboard.launch(share=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [9]:
import os
import hmac
from functools import wraps
from flask import request, jsonify

# Set this secret before running the app.
# For a demo, configure it privately in Colab.
ADMIN_PASSWORD = os.environ.get("WALLET_ADMIN_PASSWORD")

def require_admin(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        supplied = request.headers.get("X-Admin-Key", "")

        if not ADMIN_PASSWORD or not hmac.compare_digest(
            supplied, ADMIN_PASSWORD
        ):
            return jsonify({"error": "Admin access denied"}), 401

        return f(*args, **kwargs)

    return decorated

In [7]:
# Check whether the registration route exists
routes = [rule.rule for rule in app.url_map.iter_rules()]
print("Registration route exists:", "/register" in routes)

# Test registration with a new username
response = client.post("/register", json={
    "username": "testuser2026",
    "password": "TestStrongPass123!"
})

print("Status code:", response.status_code)
print("Response:", response.get_json(silent=True))
print("Raw response:", response.get_data(as_text=True))

Registration route exists: True


NameError: name 'client' is not defined

In [10]:

client = app.test_client()

# Register Alice
response = client.post("/register", json={
    "username": "alice",
    "password": "AliceStrongPass123!"
})
print("Alice registration:", response.status_code)
alice_token = response.get_json()["access_token"]

# Register Bob
response = client.post("/register", json={
    "username": "bob",
    "password": "BobStrongPass123!"
})
print("Bob registration:", response.status_code)
bob_token = response.get_json()["access_token"]

alice_headers = {"Authorization": f"Bearer {alice_token}"}
bob_headers = {"Authorization": f"Bearer {bob_token}"}

print("Alice:", client.get(
    "/balance", headers=alice_headers
).get_json())

print("Bob:", client.get(
    "/balance", headers=bob_headers
).get_json())


Alice registration: 201
Bob registration: 201
Alice: {'balance': '0.00', 'currency': 'INR', 'username': 'alice'}
Bob: {'balance': '0.00', 'currency': 'INR', 'username': 'bob'}


In [11]:

# Add ₹1,000 to Alice's demo account.
demo_fund("alice", "1000.00")

# Transfer ₹250.50 from Alice to Bob.
response = client.post(
    "/transfer",
    headers=alice_headers,
    json={
        "receiver": "bob",
        "amount": "250.50"
    }
)

print("Transfer result:", response.status_code)
print(response.get_json())

# Check updated balances.
print("Alice's balance:", client.get(
    "/balance", headers=alice_headers
).get_json())

print("Bob's balance:", client.get(
    "/balance", headers=bob_headers
).get_json())


Transfer result: 201
{'amount': '250.50', 'currency': 'INR', 'message': 'Transfer successful', 'receiver': 'bob', 'sender': 'alice', 'transaction_id': 1}
Alice's balance: {'balance': '749.50', 'currency': 'INR', 'username': 'alice'}
Bob's balance: {'balance': '250.50', 'currency': 'INR', 'username': 'bob'}


In [12]:

response = client.get(
    "/transactions?limit=20&offset=0",
    headers=alice_headers
)

print("Transaction history:")
for transaction in response.get_json()["transactions"]:
    print(transaction)


Transaction history:
{'amount': '250.50', 'created_at': '2026-10-14 09:39:21', 'currency': 'INR', 'direction': 'sent', 'receiver': 'bob', 'sender': 'alice', 'transaction_id': 1}


In [13]:

# Attempt a transfer without authentication.
response = client.post("/transfer", json={
    "receiver": "bob",
    "amount": "10.00"
})
print("Unauthenticated transfer:", response.status_code)
# Expected: 401

# Attempt to transfer more than Alice's remaining balance.
response = client.post(
    "/transfer",
    headers=alice_headers,
    json={"receiver": "bob", "amount": "100000.00"}
)
print("Insufficient balance:", response.status_code)
# Expected: 400

# Attempt to transfer to the same account.
response = client.post(
    "/transfer",
    headers=alice_headers,
    json={"receiver": "alice", "amount": "10.00"}
)
print("Self-transfer:", response.status_code)
# Expected: 400

# Confirm the service health endpoint.
print("Health check:", client.get("/health").get_json())


Unauthenticated transfer: 401
Insufficient balance: 400
Self-transfer: 400
Health check: {'status': 'ok'}


In [7]:

!pip -q install gradio pytest "psycopg[binary]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 12.4 MB/s eta 0:00:00


In [14]:

import gradio as gr

def register_ui(username, password):
    response = client.post("/register", json={
        "username": username,
        "password": password
    })
    return response.get_json()

def login_ui(username, password):
    response = client.post("/login", json={
        "username": username,
        "password": password
    })
    data = response.get_json()
    if response.status_code == 200:
        return data["access_token"], "Login successful"
    return "", data.get("error", "Login failed")

def balance_ui(token):
    if not token:
        return "Please log in first."
    response = client.get(
        "/balance",
        headers={"Authorization": f"Bearer {token}"}
    )
    return response.get_json()

def transfer_ui(token, receiver, amount):
    if not token:
        return {"error": "Please log in first."}
    response = client.post(
        "/transfer",
        headers={"Authorization": f"Bearer {token}"},
        json={"receiver": receiver, "amount": amount}
    )
    return response.get_json()

def history_ui(token):
    if not token:
        return {"error": "Please log in first."}
    response = client.get(
        "/transactions",
        headers={"Authorization": f"Bearer {token}"}
    )
    return response.get_json()

with gr.Blocks(title="Secure Wallet") as demo:
    gr.Markdown("# Secure Wallet")
    gr.Markdown("Educational banking simulator — INR")

    token = gr.State("")

    with gr.Tab("Register"):
        reg_user = gr.Textbox(label="Username")
        reg_pass = gr.Textbox(label="Password", type="password")
        reg_btn = gr.Button("Create Account")
        reg_result = gr.JSON()
        reg_btn.click(
            register_ui,
            [reg_user, reg_pass],
            reg_result
        )

    with gr.Tab("Login"):
        login_user = gr.Textbox(label="Username")
        login_pass = gr.Textbox(label="Password", type="password")
        login_btn = gr.Button("Login")
        login_result = gr.Textbox(label="Status")
        login_btn.click(
            login_ui,
            [login_user, login_pass],
            [token, login_result]
        )

    with gr.Tab("Balance"):
        balance_btn = gr.Button("Check Balance")
        balance_result = gr.JSON()
        balance_btn.click(balance_ui, [token], balance_result)

    with gr.Tab("Transfer"):
        receiver = gr.Textbox(label="Receiver username")
        amount = gr.Number(label="Amount in INR", precision=2)
        transfer_btn = gr.Button("Transfer")
        transfer_result = gr.JSON()
        transfer_btn.click(
            transfer_ui,
            [token, receiver, amount],
            transfer_result
        )

    with gr.Tab("Transaction History"):
        history_btn = gr.Button("Refresh History")
        history_result = gr.JSON()
        history_btn.click(history_ui, [token], history_result)

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88dcbcd908b42fdf37.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
import os
import hmac
from functools import wraps
from flask import request, jsonify

# Set this secret before running the app.
# For a demo, configure it privately in Colab.
ADMIN_PASSWORD = os.environ.get("WALLET_ADMIN_PASSWORD")

def require_admin(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        supplied = request.headers.get("X-Admin-Key", "")

        if not ADMIN_PASSWORD or not hmac.compare_digest(
            supplied, ADMIN_PASSWORD
        ):
            return jsonify({"error": "Admin access denied"}), 401

        return f(*args, **kwargs)

    return decorated

In [16]:
import getpass
os.environ["WALLET_ADMIN_PASSWORD"] = getpass.getpass(
    "Enter a private admin password: "
)
ADMIN_PASSWORD = os.environ["WALLET_ADMIN_PASSWORD"]

Enter a private admin password: ··········


In [1]:
import os
import secrets
import hmac
import sqlite3

from flask import Flask, request, jsonify

# Create the Flask application
app = Flask(__name__)

# Generate an administrator key for this demo
if not os.environ.get("WALLET_ADMIN_KEY"):
    os.environ["WALLET_ADMIN_KEY"] = secrets.token_urlsafe(32)

ADMIN_KEY = os.environ["WALLET_ADMIN_KEY"]

print("Flask app created successfully.")
print("Administrator key configured.")

Flask app created successfully.
Administrator key configured.


In [3]:
import os
import sqlite3
import hmac
import secrets
from functools import wraps
from flask import Flask, request, jsonify

# Create the Flask application ONCE
app = Flask(__name__)

# Configure demo administrator key
ADMIN_KEY = os.environ.get("WALLET_ADMIN_KEY")
if not ADMIN_KEY:
    ADMIN_KEY = secrets.token_urlsafe(32)
    os.environ["WALLET_ADMIN_KEY"] = ADMIN_KEY

print("Flask application created.")
print("Admin key configured.")

Flask application created.
Admin key configured.


In [7]:

import os
import hmac
import secrets
from functools import wraps
from flask import request, jsonify

# Configure a private demo admin key.
# In production, use secure credential storage and proper admin login.
if not os.environ.get("WALLET_ADMIN_KEY"):
    os.environ["WALLET_ADMIN_KEY"] = secrets.token_urlsafe(32)

ADMIN_KEY = os.environ["WALLET_ADMIN_KEY"]


def require_admin(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        supplied_key = request.headers.get("X-Admin-Key", "")

        if not hmac.compare_digest(supplied_key, ADMIN_KEY):
            return jsonify({"error": "Unauthorized"}), 401

        return f(*args, **kwargs)
    return decorated


@app.get("/admin/dashboard")
@require_admin
def admin_dashboard_api():
    try:
        users = db.execute(
            "SELECT COUNT(*) FROM users"
        ).fetchone()[0]

        accounts = db.execute(
            "SELECT COUNT(*) FROM accounts"
        ).fetchone()[0]

        transactions = db.execute(
            "SELECT COUNT(*) FROM transactions"
        ).fetchone()[0]

        return jsonify({
            "total_users": users,
            "total_accounts": accounts,
            "total_transactions": transactions
        })

    except Exception:
        app.logger.exception("Dashboard summary failed")
        return jsonify({"error": "Dashboard unavailable"}), 500


@app.get("/admin/users")
@require_admin
def admin_users_api():
    try:
        # Never expose password hashes or authentication tokens.
        rows = db.execute("""
            SELECT u.id, u.username
            FROM users u
            ORDER BY u.id DESC
            LIMIT 500
        """).fetchall()

        return jsonify({
            "users": [dict(row) for row in rows]
        })

    except Exception:
        app.logger.exception("User listing failed")
        return jsonify({"error": "Unable to list users"}), 500


@app.get("/admin/transactions")
@require_admin
def admin_transactions_api():
    try:
        rows = db.execute("""
            SELECT transaction_id, sender, receiver,
                   amount, currency, created_at
            FROM transactions
            ORDER BY transaction_id DESC
            LIMIT 500
        """).fetchall()

        return jsonify({
            "transactions": [dict(row) for row in rows]
        })

    except Exception:
        app.logger.exception("Transaction listing failed")
        return jsonify({"error": "Unable to list transactions"}), 500

print("Admin routes registered.")


Admin routes registered.


In [8]:

import gradio as gr
import pandas as pd

# Use your existing Flask application.
client = app.test_client()


def admin_request(path, admin_key):
    response = client.get(
        path,
        headers={"X-Admin-Key": admin_key}
    )

    if response.status_code != 200:
        return None, (
            f"Request failed ({response.status_code}): "
            f"{response.get_json(silent=True)}"
        )

    return response.get_json(), "Data refreshed successfully."


def load_summary(admin_key):
    data, error = admin_request("/admin/dashboard", admin_key)
    if data is None:
        return error

    return (
        "## Admin Dashboard\n"
        f"- **Registered users:** {data['total_users']}\n"
        f"- **Accounts:** {data['total_accounts']}\n"
        f"- **Transactions:** {data['total_transactions']}"
    )


def load_users(admin_key):
    data, error = admin_request("/admin/users", admin_key)
    if data is None:
        return pd.DataFrame(), error

    return pd.DataFrame(data["users"]), error


def load_transactions(admin_key):
    data, error = admin_request("/admin/transactions", admin_key)
    if data is None:
        return pd.DataFrame(), error

    return pd.DataFrame(data["transactions"]), error


with gr.Blocks(title="Wallet Admin Dashboard") as admin_ui:
    gr.Markdown("# Wallet Admin Dashboard")
    gr.Markdown(
        "Manage visibility of users and monitor wallet activity."
    )

    admin_key = gr.Textbox(
        label="Administrator key",
        type="password"
    )

    with gr.Tab("Overview"):
        summary = gr.Markdown()
        refresh_summary = gr.Button("Refresh Overview")
        refresh_summary.click(
            load_summary,
            inputs=admin_key,
            outputs=summary
        )

    with gr.Tab("Users"):
        users_table = gr.Dataframe(
            label="Registered Users",
            interactive=False
        )
        users_status = gr.Textbox(label="Status", interactive=False)
        refresh_users = gr.Button("Load Users")
        refresh_users.click(
            load_users,
            inputs=admin_key,
            outputs=[users_table, users_status]
        )

    with gr.Tab("Transactions"):
        transactions_table = gr.Dataframe(
            label="Recent Transactions",
            interactive=False
        )
        transactions_status = gr.Textbox(
            label="Status", interactive=False
        )
        refresh_transactions = gr.Button("Load Transactions")
        refresh_transactions.click(
            load_transactions,
            inputs=admin_key,
            outputs=[transactions_table, transactions_status]
        )

admin_ui.launch(share=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [9]:
# Launch the Gradio admin dashboard
admin_ui.launch(
    share=True,
    debug=True
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://46e74ed126365f941c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://46e74ed126365f941c.gradio.live


In [10]:

import gradio as gr
import pandas as pd

# Requires your existing Flask test client named `client`.
# The user and admin API routes must already be registered.

def show_user_portal():
    return (
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False)
    )

def show_admin_portal():
    return (
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=True)
    )

def go_home():
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False)
    )

def register_user(username, password):
    response = client.post("/register", json={
        "username": username,
        "password": password
    })
    return f"Status: {response.status_code}\n{response.get_json(silent=True)}"

def login_user(username, password):
    response = client.post("/login", json={
        "username": username,
        "password": password
    })
    data = response.get_json(silent=True) or {}

    if response.status_code == 200 and data.get("access_token"):
        return data["access_token"], "Login successful."

    return "", f"Login failed: {data}"

def check_balance(token):
    response = client.get(
        "/balance",
        headers={"Authorization": f"Bearer {token}"}
    )
    return str(response.get_json(silent=True))

def transfer_money(token, receiver, amount):
    response = client.post(
        "/transfer",
        json={"receiver": receiver, "amount": amount},
        headers={"Authorization": f"Bearer {token}"}
    )
    return f"Status: {response.status_code}\n{response.get_json(silent=True)}"

def get_history(token):
    response = client.get(
        "/transactions?limit=20&offset=0",
        headers={"Authorization": f"Bearer {token}"}
    )
    data = response.get_json(silent=True) or {}
    return pd.DataFrame(data.get("transactions", []))

def admin_summary(key):
    response = client.get(
        "/admin/dashboard",
        headers={"X-Admin-Key": key}
    )
    return str(response.get_json(silent=True))

def admin_users(key):
    response = client.get(
        "/admin/users",
        headers={"X-Admin-Key": key}
    )
    data = response.get_json(silent=True) or {}
    return pd.DataFrame(data.get("users", [])), str(data)

def admin_transactions(key):
    response = client.get(
        "/admin/transactions",
        headers={"X-Admin-Key": key}
    )
    data = response.get_json(silent=True) or {}
    return pd.DataFrame(data.get("transactions", [])), str(data)


with gr.Blocks(title="Smart Wallet Management System") as portal_app:

    # COVER PAGE
    with gr.Column(visible=True) as home:
        gr.Markdown("""
        # 💳 SMART WALLET MANAGEMENT SYSTEM

        ### Secure Digital Transaction Management

        Welcome to the unified wallet platform.

        Select a portal to continue.
        """)

        with gr.Row():
            user_button = gr.Button(
                "👤 User Portal",
                variant="primary"
            )
            admin_button = gr.Button(
                "🛡️ Admin Portal",
                variant="secondary"
            )

        gr.Markdown("""
        **User services:** Registration, login, balance,
        money transfers and transaction history.

        **Admin services:** User overview, account monitoring
        and transaction review.
        """)

    # USER PORTAL
    with gr.Column(visible=False) as user_panel:
        gr.Markdown("## 👤 User Portal")
        home_user = gr.Button("← Home")

        with gr.Tab("Register"):
            reg_username = gr.Textbox(label="Username")
            reg_password = gr.Textbox(
                label="Password", type="password"
            )
            reg_button = gr.Button("Register")
            reg_output = gr.Textbox(label="Registration result")

            reg_button.click(
                register_user,
                inputs=[reg_username, reg_password],
                outputs=reg_output
            )

        with gr.Tab("Login"):
            login_username = gr.Textbox(label="Username")
            login_password = gr.Textbox(
                label="Password", type="password"
            )
            login_button = gr.Button("Login")
            user_token = gr.State("")
            login_status = gr.Textbox(label="Login status")

            login_button.click(
                login_user,
                inputs=[login_username, login_password],
                outputs=[user_token, login_status]
            )

        with gr.Tab("Balance"):
            balance_button = gr.Button("Check Balance")
            balance_output = gr.Textbox(label="Wallet balance")

            balance_button.click(
                check_balance,
                inputs=user_token,
                outputs=balance_output
            )

        with gr.Tab("Transfer"):
            receiver = gr.Textbox(label="Receiver username")
            amount = gr.Textbox(label="Amount in INR")
            transfer_button = gr.Button("Transfer")
            transfer_output = gr.Textbox(label="Transfer result")

            transfer_button.click(
                transfer_money,
                inputs=[user_token, receiver, amount],
                outputs=transfer_output
            )

        with gr.Tab("History"):
            history_button = gr.Button("Load Transactions")
            history_output = gr.Dataframe(
                label="My Transaction History"
            )

            history_button.click(
                get_history,
                inputs=user_token,
                outputs=history_output
            )

    # ADMIN PORTAL
    with gr.Column(visible=False) as admin_panel:
        gr.Markdown("## 🛡️ Admin Portal")
        home_admin = gr.Button("← Home")

        admin_key = gr.Textbox(
            label="Administrator key",
            type="password"
        )

        with gr.Tab("Overview"):
            summary_button = gr.Button("Load Overview")
            summary_output = gr.Textbox(
                label="Dashboard Summary"
            )
            summary_button.click(
                admin_summary,
                inputs=admin_key,
                outputs=summary_output
            )

        with gr.Tab("Users"):
            users_button = gr.Button("Load Users")
            users_table = gr.Dataframe(
                label="Registered Users"
            )
            users_status = gr.Textbox(label="Status")
            users_button.click(
                admin_users,
                inputs=admin_key,
                outputs=[users_table, users_status]
            )

        with gr.Tab("Transactions"):
            transactions_button = gr.Button(
                "Load Transactions"
            )
            transactions_table = gr.Dataframe(
                label="Transaction Records"
            )
            transactions_status = gr.Textbox(label="Status")
            transactions_button.click(
                admin_transactions,
                inputs=admin_key,
                outputs=[transactions_table, transactions_status]
            )

    user_button.click(
        show_user_portal,
        outputs=[home, user_panel, admin_panel]
    )
    admin_button.click(
        show_admin_portal,
        outputs=[home, user_panel, admin_panel]
    )
    home_user.click(
        go_home,
        outputs=[home, user_panel, admin_panel]
    )
    home_admin.click(
        go_home,
        outputs=[home, user_panel, admin_panel]
    )

portal_app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0e632d553e06543f83.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
db = sqlite3.connect(":memory:", check_same_thread=False)
db.row_factory = sqlite3.Row

print("SQLite database initialized.")

SQLite database initialized.


In [5]:
client = app.test_client()

print("Available routes:")
for rule in app.url_map.iter_rules():
    print(rule.rule)

Available routes:
/static/<path:filename>


In [2]:
@app.post("/admin/deposit")
def admin_deposit():
    supplied_key = request.headers.get("X-Admin-Key", "")

    if not hmac.compare_digest(supplied_key, ADMIN_KEY):
        return jsonify({"error": "Unauthorized"}), 401

    data = request.get_json(silent=True) or {}

    username = data.get("username")
    amount = data.get("amount")

    if not username or amount is None:
        return jsonify({
            "error": "username and amount are required"
        }), 400

    # This route requires your existing database and helper functions.
    # Do not run it until the database schema and account table exist.
    return jsonify({
        "message": "Admin route is registered.",
        "username": username,
        "note": "Connect this route to your existing account database."
    }), 200

print("Admin deposit route registered.")

Admin deposit route registered.
